# Regime-Adaptive Portfolio Analysis

Interactive exploration notebook. All computation imports from `src/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import config
from src.data.fetcher import fetch_etf_prices, fetch_fama_french, compute_log_returns

# Load data
prices = pd.read_csv('../data/raw/etf_prices.csv', index_col=0, parse_dates=True)
ff_data = pd.read_csv('../data/raw/ff5_daily.csv', index_col=0, parse_dates=True)
log_ret = compute_log_returns(prices)

print(f'Data: {prices.shape[0]} days, {prices.shape[1]} tickers')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')

In [ ]:
# Run walk-forward backtest
from src.backtest.walk_forward import WalkForwardEngine
from src.backtest.metrics import compute_metrics, metrics_to_df

engine = WalkForwardEngine()
results = engine.run(prices, ff_data=ff_data)

# Compute metrics
m_strategy = compute_metrics(results['strategy_returns'],
                              stress_signals=results['stress_signals'],
                              turnover_series=results['turnover'])
m_benchmark = compute_metrics(results['benchmark_returns'])
m_ew = compute_metrics(results['equal_weight_returns'])

metrics_to_df({'Strategy': m_strategy, 'SPY': m_benchmark, 'Equal-Weight': m_ew})

In [ ]:
# Generate figures
from src.visualization.plots import (
    plot_regime_detection, plot_backtest, plot_detector_heatmap,
    plot_rolling_sharpe, plot_membership_functions
)

plot_backtest(results['strategy_returns'], results['benchmark_returns'],
              results['equal_weight_returns'], m_strategy, save=False)
plt.show()